In [11]:
from z3 import *

s=Solver()
# https://philosophy.hku.hk/think/logic/knights.php
# 382
# A very special island is inhabited only by knights and knaves. Knights always tell the truth, and knaves always lie.
# You meet nine inhabitants: Mel, Bart, Sue, John, Rex, David, Tom, Zoey and Homer. 

# Mel claims that only a knave would say that Tom is a knave. 
# Bart claims, “Rex is a knave.” 
# Sue says that Mel and Homer are knaves. 
# John tells you, “I know that I am a knight and that Tom is a knave.” 
# Rex says, “John and I are both knights.” 
# David tells you that at least one of the following is true: that Tom is a knight or that Sue is a knight. 
# Tom says, “It's false that John is a knave.” 
# Zoey says, “It's not the case that Sue is a knave.” 
# Homer tells you that John is a knave or David is a knave.”

# Can you determine who is a knight and who is a knave?

M, B, S, J, R, D, T, Z, H = Bools("M B S J R D T Z H")

conds=[
    M==(False==(T==False)), # if M True then statment inside () also true if M is False then his statement also False 
    B==(R==False),
    S==(And(M==False,H==False)),
    J==(And(J==True,T==False)),
    R==(And(J==True,R==True)),
    D==(Or(T==True,S==True)),
    T==((J==False)==False),
    Z==((S==False)==False),
    H==(Or(J==False,D==False))
]

s.add(conds)

while True:
    st=s.check()
    if st==sat:
        m = s.model()
        print(m)
        s.add(Not(And([m.eval(x)==x  for x in (M, B, S, J, R, D, T, Z, H) ])))
    else:
        print(st)
        break



[D = False,
 S = False,
 J = False,
 B = True,
 M = False,
 H = True,
 Z = False,
 R = False,
 T = False]
unsat


In [12]:
from z3 import *

# A very special island is inhabited only by knights and knaves. Knights always tell the truth, and knaves always lie.

# You meet nine inhabitants: Bozo, Zoey, Marge, Tom, Anna, Joe, Dave, Sue and Rex. 

# Bozo claims that it's false that Dave is a knave. 
# Zoey tells you that Marge is a knave and Sue is a knight. 
# Marge claims that Bozo is a knave. 
# Tom claims, “Anna could claim that I am a knight.” 
# Anna says that only a knave would say that Dave is a knave. 
# Joe says, “It's not the case that Tom is a knave.” 
# Dave claims that Rex and Joe are both knights or both knaves. 
# Sue tells you that Anna is a knave. 
# Rex claims, “Neither Bozo nor Zoey are knaves.”

# Can you determine who is a knight and who is a knave?

s=Solver()
B,Z,M,T,A,J,D,S,R = Bools("B Z M T A J D S R")

conds=[
    B==(False==(D==False)), 
    Z==(And(M==False,S==True)),
    M==(B==False),
    T==(A==(T==True)),
    A==(False==(D==False)),
    J==(False==(T==False)),
    D==(J==R),
    S==(A==False),
    R==(And(B==True,Z==True))
]

s.add(conds)

while True:
    st=s.check()
    if st==sat:
        m = s.model()
        print(m)
        s.add(Not(And([m.eval(x)==x  for x in (M, B, S, J, R, D, T, Z, H) ])))
    else:
        print(st)
        break


[D = True,
 S = False,
 A = True,
 B = True,
 J = False,
 M = False,
 Z = False,
 R = False,
 T = False]
unsat


In [13]:
from z3 import *

s=Solver()
#1 fair 1 lier 1 question
peoples = [ Bool("people_%s" % (j+1)) for j in range(2)  ]


question = Function("question", BoolSort(),BoolSort(),  BoolSort()) 
determine_people = Function("determine_people", BoolSort(),  BoolSort()) 


#s.add(ForAll(peoples,And(question(peoples[0],peoples[1])==peoples[0])))

x,y=Bools("x y")


s.add(ForAll(peoples+[y],Implies(And(Distinct(peoples)),
                                            peoples[0]
                                                        ==
                                                determine_people(
                                                    question(peoples[0],peoples[1])==peoples[0]))))

if s.check() == sat:
    m = s.model()
    print(m)


print(s.check())


[determine_people = [True -> False, else -> True],
 question = [else ->
             And(Not(And(Not(Var(0)), Var(1))),
                 Not(And(Var(0), Not(Var(1)))))]]
sat


In [16]:
from z3 import *

s=Solver()
# 1 fair 2 lier 2 question
fairs=1
peoples = [ Bool("people_%s" % (j+1)) for j in range(3)  ]

who_to_ask1 = Function("who_to_ask1", IntSort()) 
who_to_ask2 = Function("who_to_ask2", BoolSort(),IntSort()) 
question1 = Function("question1", BoolSort(),BoolSort(),BoolSort(),  BoolSort()) 
question2 = Function("question2", BoolSort(),BoolSort(),BoolSort(),  BoolSort()) 
determine_people = Function("determine_people", BoolSort(),BoolSort(),IntSort(),  BoolSort()) 

x,y=Bools("x y")
xi,yi=Ints("xi yi")
s.add(0<=who_to_ask1(),who_to_ask1()<3)
s.add(ForAll([x],And(0<=who_to_ask2(x),who_to_ask2(x)<3)))

def GetArrayElement(arr,index):
    current_else = arr[0]
    for i in range(1,len(arr)):
        current_else = If(index==i,arr[i],current_else)      
    return current_else

def PrintFunc(m,func,args,current_val=[]):
    if len(args)==0:
        print(current_val,m.eval(func(*current_val)))
        return
    for value in args[0]:
        PrintFunc(m,func,args[1:],current_val+[value])
            


answer1 = question1(peoples[0],peoples[1],peoples[2])==GetArrayElement(peoples,who_to_ask1())
answer2 = question2(peoples[0],peoples[1],peoples[2])==GetArrayElement(peoples,who_to_ask2(answer1))

s.add(ForAll(peoples+[yi],Implies(And(Sum(peoples)==fairs,0<=yi,yi<3),
                                            GetArrayElement(peoples,yi) == determine_people(answer1,answer2,yi))))

if s.check() == sat:
    m = s.model()
    print("determine_people")
    PrintFunc(m,determine_people,[(False,True),(False,True),(0,1,2)])
    print("who_to_ask1")
    PrintFunc(m,who_to_ask1,[])
    print("who_to_ask2")
    PrintFunc(m,who_to_ask2,[(False,True)])
    print("question1")
    PrintFunc(m,question1,[(False,True),(False,True),(False,True)])
    print("question2")
    PrintFunc(m,question2,[(False,True),(False,True),(False,True)])

print(s.check())


determine_people
[False, False, 0] False
[False, False, 1] False
[False, False, 2] True
[False, True, 0] False
[False, True, 1] True
[False, True, 2] False
[True, False, 0] True
[True, False, 1] False
[True, False, 2] False
[True, True, 0] False
[True, True, 1] False
[True, True, 2] False
who_to_ask1
[] 2
who_to_ask2
[False] 0
[True] 0
question1
[False, False, False] True
[False, False, True] False
[False, True, False] True
[False, True, True] True
[True, False, False] False
[True, False, True] True
[True, True, False] True
[True, True, True] True
question2
[False, False, False] False
[False, False, True] True
[False, True, False] False
[False, True, True] False
[True, False, False] False
[True, False, True] False
[True, True, False] False
[True, True, True] False
sat


In [52]:
from z3 import *
from pprint import pprint
import math

def GetArrayElement(arr,index):#TODO optimize code
    current_else = arr[0]
    for i in range(1,len(arr)):
        current_else = If(index==i,arr[i],current_else)
    return current_else

def PrintFunc(m,func,args,current_val=[]):
    if current_val==[]:
        print(func.name())
    if len(args)==0:
        print(current_val,m.eval(func(*current_val)))
        return
    for value in args[0]:
        PrintFunc(m,func,args[1:],current_val+[value])

def solve(Liers,Fairs,min_questions=-1,language_unknown=False):
    if min_questions==-1:
        min_questions=math.log2(math.factorial(Liers+Fairs)/math.factorial(Fairs)/math.factorial(Liers)) #TODO not able to calculate solve(3,3) and solve(2,5) without this hint,it take very long time to prove that min_questions-1 is not suffiecient number of questions
        min_questions=math.ceil(min_questions)
    for Q in range(min_questions,min_questions+1):#if found then it is optimal if return -1 then delete hints and increase limit of range

        s=Solver()
        N=Liers+Fairs
        peoples = [ Bool("people_%s" % (j+1)) for j in range(N)  ]
        who_to_ask = [Function("who_to_ask_%i" % i,*([BoolSort()]*i), IntSort()) for i in range(Q)]
        question = [Function("question_%i" % i,*([BoolSort()]*N), BoolSort()) for i in range(Q)]
        determine_people = Function("determine_people", *([BoolSort()]*Q),IntSort(),  BoolSort())

        yes_answer=Bool("yes")
        no_answer=Bool("no")
        lang=[yes_answer,no_answer]

        # x = [Bool("x_%i" % i) for i in range(Q)]
        # for i in range(Q):
        #     args=x[:i]
        #     if len(args)==0:
        #         s.add(And(0<=who_to_ask[i](),who_to_ask[i]()<N))
        #     else:
        #         s.add(ForAll(args,And(0<=who_to_ask[i](*args),who_to_ask[i](*args)<N)))

        # answers=[]
        # for i in range(Q):
        #     answers.append(question[i](*peoples)==GetArrayElement(peoples,who_to_ask[i](*answers))) 

        answers=[False==peoples[0]]# assuming that first question will be always False which will determine exact type of first people
        for i in range(1,Q):
            answers.append(question[i](*peoples)==peoples[0])#==GetArrayElement(peoples,who_to_ask[i](*answers))) #works way faster if we assume that we always wil ask questions to one of peoples

        y=Int("y")
        s.add(ForAll(peoples+[y],Implies(And(Sum(peoples)==Fairs,0<=y,y<N),
                                                    GetArrayElement(peoples,y) == determine_people(*answers,y))))

        if s.check() == sat:
            m = s.model()
            return m,determine_people,who_to_ask,question
        else:
            continue
    return -1


#able to calculate without hint from information theory 
   #0 #1 #2 #3 #4 #5 #6 #7 #8 #9 #10#11#12#13#14#15#16
#0 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
#1 [0, 1, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 5]
#2 [0, 2, 3, 4, 4, -, -, -, -, -, -, -, -, -, -, -, -]
#3 [0, 2, 4, -, -, -, -, -, -, -, -, -, -, -, -, -, -]
#4 [0, 3, 4, -, -, -, -, -, -, -, -, -, -, -, -, -, -]
#5 [0, 3, -, -, -, -, -, -, -, -, -, -, -, -, -, -, -]
#6 [0, 3, -, -, -, -, -, -, -, -, -, -, -, -, -, -, -]
#7 [0, 3, -, -, -, -, -, -, -, -, -, -, -, -, -, -, -]
#8 [0, 4, -, -, -, -, -, -, -, -, -, -, -, -, -, -, -]
#9 [0, 4, -, -, -, -, -, -, -, -, -, -, -, -, -, -, -]


#able to calculate with hint from information theory 
   #0 #1 #2 #3 #4 #5 #6 #7 #8 #9 #10#11#12#13#14#15#16
#0 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
#1 [0, 1, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 5]
#2 [0, 2, 3, 4, 4, 5, 5, 6, 6, 6, 7, 7, 7, 7, 7, 8, 8]
#3 [0, 2, 4, 5, 6, 6, 7, 7, 8, 8, 9, 9, -, -, -, -, -]
#4 [0, 3, 4, 6, 7, 7, 8, 9, -, -, -, -, -, -, -, -, -]
#5 [0, 3, 5, 6, 7, 8, 9, -, -, -, -, -, -, -, -, -, -]
#6 [0, 3, 5, 7, 8, 9,10, -, -, -, -, -, -, -, -, -, -]
#7 [0, 3, 6, 7, 9, -, -, -, -, -, -, -, -, -, -, -, -]
#8 [0, 4, 6, 8, -, -, -, -, -, -, -, -, -, -, -, -, -]
#9 [0, 4, 6, 8, -, -, -, -, -, -, -, -, -, -, -, -, -]


Liers=1
Fairs=1
(m,determine_people,who_to_ask,question) = solve(Liers,Fairs)
Q=len(question)
PrintFunc(m,determine_people,[(False,True)]*Q+[[i for i in range(Liers+Fairs)]])
for i in range(Q):
    PrintFunc(m,who_to_ask[i],[(False,True)]*i)
    PrintFunc(m,question[i],[(False,True)]*(Liers+Fairs))
print("Optimal number of questions is "+str(Q))

n=8
table=[0 for x in range(n)]
for x in range(8,n):
    (m,determine_people,who_to_ask,question) = solve(x,4)
    print(x,len(question))

determine_people
[False, 0] True
[False, 1] False
[True, 0] False
[True, 1] True
who_to_ask_0
[] who_to_ask_0
question_0
[False, False] question_0(False, False)
[False, True] question_0(False, True)
[True, False] question_0(True, False)
[True, True] question_0(True, True)
Optimal number of questions is 1


In [56]:
from z3 import *

s=Solver()
# 1 fair 1 lier 2 question
# unknown language 
fairs=1
num_people=2
peoples = [ Bool("people_%s" % (j+1)) for j in range(num_people)  ]

who_to_ask1 = Function("who_to_ask1", IntSort()) 
who_to_ask2 = Function("who_to_ask2", IntSort()) 
question1 = Function("question1", BoolSort(),BoolSort(),  BoolSort(),BoolSort(),  BoolSort()) 
question2 = Function("question2", BoolSort(),BoolSort(),  BoolSort(),BoolSort(),  BoolSort()) 
determine_people = Function("determine_people", BoolSort(),IntSort(),  BoolSort()) 

x,y=Bools("x y")
xi,yi=Ints("xi yi")
s.add(0<=who_to_ask1(),who_to_ask1()<num_people)
s.add(And(0<=who_to_ask2(),who_to_ask2()<num_people))

def GetArrayElement(arr,index):
    current_else = arr[0]
    for i in range(1,len(arr)):
        current_else = If(index==i,arr[i],current_else)      
    return current_else

def PrintFunc(m,func,args,current_val=[]):
    if len(args)==0:
        print(current_val,m.eval(func(*current_val)))
        return
    for value in args[0]:
        PrintFunc(m,func,args[1:],current_val+[value])
            
yes_answer=Bool("yes")
no_answer=Bool("no")

answer1 = (question1(peoples[0],peoples[1],yes_answer,no_answer)==yes_answer)==GetArrayElement(peoples,who_to_ask1())
answer2 = (question2(peoples[0],peoples[1],yes_answer,no_answer)==yes_answer)==GetArrayElement(peoples,who_to_ask2())

s.add(ForAll(peoples+[yi,yes_answer,no_answer],Implies(And(Sum(peoples)==fairs,Distinct(yes_answer,no_answer),0<=yi,yi<num_people),
                                            GetArrayElement(peoples,yi) == determine_people(answer1==answer2,yi))))

if s.check() == sat:
    m = s.model()
    print("determine_people")
    PrintFunc(m,determine_people,[(False,True),(0,1)])
    print("who_to_ask1")
    PrintFunc(m,who_to_ask1,[])
    print("who_to_ask2")
    PrintFunc(m,who_to_ask2,[])
    print("question1")
    PrintFunc(m,question1,[(False,True),(False,True),(False,True),(False,True)])
    print("question2")
    PrintFunc(m,question2,[(False,True),(False,True),(False,True),(False,True)])

print(s.check())

#we can ask "do you fair and answer of that question corresponds to NO" question to first one
#we can ask "do you fair and answer of that question corresponds to YES" question to second one
#if they answer same way then its mean that first one is lier and second one fair
#if they answer different way then its mean that first one is fair and second one lier

determine_people
[False, 0] False
[False, 1] True
[True, 0] True
[True, 1] False
who_to_ask1
[] 0
who_to_ask2
[] 0
question1
[False, False, False, False] True
[False, False, False, True] True
[False, False, True, False] True
[False, False, True, True] True
[False, True, False, False] True
[False, True, False, True] True
[False, True, True, False] False
[False, True, True, True] True
[True, False, False, False] True
[True, False, False, True] True
[True, False, True, False] False
[True, False, True, True] True
[True, True, False, False] True
[True, True, False, True] True
[True, True, True, False] True
[True, True, True, True] True
question2
[False, False, False, False] True
[False, False, False, True] True
[False, False, True, False] True
[False, False, True, True] True
[False, True, False, False] True
[False, True, False, True] False
[False, True, True, False] True
[False, True, True, True] True
[True, False, False, False] True
[True, False, False, True] True
[True, False, True, False